# 03 · Are cheaper models right for the right reason?

For every test-set jellyfish we compute Grad-CAM, Grad-CAM++ and Score-CAM for every model, then score each map against the **simulation ground truth**:

| metric | meaning |
|---|---|
| **TEF** tail energy fraction | share of the heatmap on extra-planar gas (tail region) |
| **TEL** tail energy lift | TEF ÷ tail area fraction (1 = no better than a uniform map) |
| **GEF** gas energy fraction | share of the heatmap on any gas (vs empty background) |
| **PG** pointing game | heatmap maximum lies on gas |
| **DEL** deletion AUC | P(jellyfish) as the top-ranked pixels are removed (lower = more faithful) |
| **AGR** agreement | Spearman correlation with the ResNet-50 map for the same galaxy and seed |

We also measure the **energy cost per 1,000 explanations**, which is rarely reported.

In [1]:
import sys, warnings; from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src')); warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, json
%matplotlib inline

In [2]:
import train, xai
from models import MODELS, PRETTY, build
from data import GasMapDataset
from energy import measure_idle
measure_idle(10)
df = pd.read_csv(ROOT/'data/processed/splits.csv')
te = GasMapDataset(df[df.split=='test'])
jf = te.df.label.values == 1
x = torch.from_numpy(te.x[jf]); rows_jf = list(te.df[jf].itertuples())
print(len(x), 'test jellyfish')

148 test jellyfish


In [3]:
# Memory-light and resumable: one (seed, model) at a time; per-run results saved to results/xai_parts/.
# Only seed-0 Grad-CAM maps are kept (for the figure), as float16. Finished runs are not recomputed.
import gc
PART = ROOT/'results/xai_parts'; PART.mkdir(parents=True, exist_ok=True)
SEEDS = [0, 1, 2]; cams_seed0 = {}
for seed in SEEDS:
    todo = [m for m in MODELS if not (PART/f'{m}_s{seed}.csv').exists()]
    ref = {}  # ResNet-50 maps for this seed (agreement reference)
    for name in ['resnet50'] + [m for m in MODELS if m != 'resnet50']:
        f = PART/f'{name}_s{seed}.csv'; done = f.exists()
        methods = list(xai.METHODS) if not done else \
                  (list(xai.METHODS) if (name == 'resnet50' and todo) else (['gradcam'] if seed == 0 else []))
        if not methods:
            continue
        m = build(name, pretrained=False).to(train.DEVICE)
        m.load_state_dict(torch.load(ROOT/f'checkpoints/{name}_s{seed}.pt', map_location=train.DEVICE)); m.eval()
        probs = train.predict(m, te)[jf]; rows = []
        for meth in methods:
            c = xai.compute_cams(m, name, meth, x)
            if name == 'resnet50': ref[meth] = c
            if seed == 0 and meth == 'gradcam': cams_seed0[(name, meth, 0)] = c.astype(np.float16)
            if done:
                continue
            d, _ = xai.deletion_auc(m, x, c)
            agr = xai.agreement(c, ref[meth])
            for i, (pm, r) in enumerate(zip(xai.physical_metrics(c, rows_jf), rows_jf)):
                rows.append(dict(model=name, method=meth, seed=seed, key=r.key, p_jf=probs[i], DEL=d[i], AGR=agr[i], **pm))
            if seed == 0:
                cf = PART/f'cost_{name}_{meth}.json'
                if not cf.exists(): cf.write_text(json.dumps(dict(model=name, method=meth, **xai.explain_cost(m, name, meth, x))))
            del c; gc.collect()
        if rows: pd.DataFrame(rows).to_csv(f, index=False)
        del m; gc.collect(); torch.mps.empty_cache()
        print(seed, PRETTY[name], 'done' if not done else 'maps only', flush=True)
X = pd.concat([pd.read_csv(f) for f in sorted(PART.glob('*_s*.csv'))])
X.to_csv(ROOT/'results/xai_per_galaxy.csv', index=False)
C = pd.DataFrame([json.loads(f.read_text()) for f in sorted(PART.glob('cost_*.json'))]); C.to_csv(ROOT/'results/xai_cost.csv', index=False)
X.groupby(['model','method'])[['TEF','TEL','GEF','PG','DEL','AGR']].mean().round(3)

100%|██████████| 128/128 [00:03<00:00, 38.75it/s]


0 ResNet-50 maps only
0 ResNet-18 maps only


100%|██████████| 80/80 [00:01<00:00, 54.04it/s]


0 EfficientNet-B0 done
0 MobileNetV3-S maps only
0 ViT-Tiny maps only
0 SmallCNN maps only


100%|██████████| 128/128 [00:03<00:00, 38.69it/s]


1 ResNet-50 maps only


... (3 outputs left out)

2 ResNet-50 maps only


100%|██████████| 80/80 [00:01<00:00, 53.61it/s]


2 EfficientNet-B0 done


TEF    TEL    GEF     PG    DEL    AGR
model                 method                                             
efficientnet_b0       gradcam    0.102  1.075  0.111  0.124  0.665  0.128
                      gradcam++  0.162  1.639  0.169  0.178  0.565  0.287
                      scorecam   0.168  2.382  0.185  0.236  0.529  0.474
mobilenetv3_small_100 gradcam    0.110  1.162  0.129  0.162  0.627  0.175
                      gradcam++  0.107  1.412  0.125  0.149  0.659  0.214
                      scorecam   0.188  2.497  0.213  0.322  0.468  0.467
resnet18              gradcam    0.204  2.566  0.226  0.462  0.609  0.565
                      gradcam++  0.210  2.704  0.233  0.500  0.591  0.688
                      scorecam   0.229  3.000  0.254  0.570  0.574  0.845
resnet50              gradcam    0.205  2.632  0.226  0.520  0.529  1.000
                      gradcam++  0.224  2.954  0.249  0.599  0.499  1.000
                      scorecam   0.262  3.573  0.292  0.597  0.409  1.000
small_cnn             gradcam    0.288  5.048  0.315  0.743  0.313  0.248
                      gradcam++  0.293  5.058  0.328  0.752  0.429  0.446
                      scorecam   0.290  5.090  0.335  0.736  0.481  0.687
vit_tiny_patch16_224  gradcam    0.117  1.615  0.130  0.297  0.601  0.088
                      gradcam++  0.119  1.780  0.133  0.153  0.699  0.227
                      scorecam   0.287  4.568  0.367  0.518  0.336  0.349

In [4]:
C.pivot(index='model', columns='method', values='xai_per1k_joules').round(1)

method,gradcam,gradcam++,scorecam
model,,,
efficientnet_b0,94.6,95.0,27449.6
mobilenetv3_small_100,24.0,26.4,3314.5
resnet18,57.7,62.6,6175.9
resnet50,142.5,145.7,66370.5
small_cnn,55.5,56.2,941.6
vit_tiny_patch16_224,25.5,26.5,1409.0


In [5]:
np.save(ROOT/'results/cams_seed0.npy', cams_seed0, allow_pickle=True)